# Wayback Machine Feasibility Check

Verify CDX coverage and HTML parseability before running a full scrape.

In [ ]:
import sys
sys.path.insert(0, '..')

from wayback_scraper.cdx import query_snapshots
from wayback_scraper.scraper import fetch_html
from wayback_scraper.extract import extract_gigs
import pandas as pd

## 1. CDX Coverage
How many Wayback snapshots exist for target categories?

In [ ]:
CATEGORIES = [
    'graphics-design',
    'digital-marketing',
    'writing-translation',
    'video-animation',
    'programming-tech',
]

coverage = {}
for cat in CATEGORIES:
    snaps = query_snapshots(f'fiverr.com/categories/{cat}*', from_year=2019, to_year=2024, limit=200)
    coverage[cat] = len(snaps)
    print(f'{cat}: {len(snaps)} snapshots')

pd.Series(coverage).sort_values(ascending=False)

## 2. Snapshot Distribution by Year

In [ ]:
# Pick one category and inspect temporal spread
snaps = query_snapshots('fiverr.com/categories/graphics-design*', from_year=2019, to_year=2024)
years = [s.year for s in snaps]
pd.Series(years).value_counts().sort_index().plot(kind='bar', title='Snapshots per year — graphics-design')

## 3. HTML Parse Test
Fetch one snapshot and check extraction yield.

In [ ]:
if snaps:
    sample = snaps[0]
    print('Fetching:', sample.wayback_url)
    html = fetch_html(sample.wayback_url)
    gigs = extract_gigs(html, category='graphics-design',
                        snapshot_timestamp=sample.timestamp,
                        wayback_url=sample.wayback_url)
    print(f'Extracted {len(gigs)} gigs')
    pd.DataFrame([g.to_dict() for g in gigs]).head()

## 4. Price Distribution Preview

In [ ]:
if gigs:
    prices = [g.price_usd for g in gigs if g.price_usd is not None]
    pd.Series(prices).describe()